In [1]:
import os, json, random
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

random.seed(42)
np.random.seed(42)

# ---------- Ingest & Clean ----------
SYNONYMS = {"IcedTea":"Iced Tea", "Cola":"Soda", "Coke":"Soda"}

def load_batches(dataset_dir, up_to_batch):
    files = [f"batch{i}.csv" for i in range(1, up_to_batch+1)]
    dfs = [pd.read_csv(os.path.join(dataset_dir, f)) for f in files]
    return pd.concat(dfs, ignore_index=True)

def parse_baskets(df):
    baskets = df["items"].fillna("").apply(lambda x: [i.strip() for i in str(x).split(",") if i.strip()]).tolist()
    clean = []
    for b in baskets:
        nb = [SYNONYMS.get(i.strip(), i.strip()) for i in b]
        clean.append(sorted(list(set(nb))))
    return clean

def onehot(baskets):
    te = TransactionEncoder()
    arr = te.fit(baskets).transform(baskets)
    return pd.DataFrame(arr, columns=te.columns_)

# ---------- Hybrid Miner (FP-Growth vs ECLAT) ----------
def eclat_frequent_itemsets(baskets, minsup):
    n = len(baskets)
    item_to_tids = {}
    for tid, basket in enumerate(baskets):
        for item in basket:
            item_to_tids.setdefault(item, set()).add(tid)

    freq = []
    items = [(frozenset([i]), tids) for i, tids in item_to_tids.items()
             if len(tids)/n >= minsup]
    items.sort(key=lambda x: len(x[1]), reverse=True)

    def extend(prefix, items_list):
        for idx in range(len(items_list)):
            itemset_i, tids_i = items_list[idx]
            sup = len(tids_i)/n
            freq.append({"support": sup, "itemsets": frozenset(list(itemset_i | prefix))})

            suffix = []
            for j in range(idx+1, len(items_list)):
                itemset_j, tids_j = items_list[j]
                inter = tids_i.intersection(tids_j)
                if len(inter)/n >= minsup:
                    suffix.append((itemset_i | itemset_j, inter))
            if suffix:
                extend(prefix, suffix)

    extend(frozenset(), items)
    df = pd.DataFrame(freq).drop_duplicates(subset=["itemsets"])
    return df.sort_values("support", ascending=False).reset_index(drop=True)

def choose_engine(onehot_df):
    density = float(onehot_df.values.mean())
    avg_basket = float(onehot_df.sum(axis=1).mean())
    if density >= 0.12 or avg_basket >= 3.2:
        return "fp_growth"
    return "eclat"

def mine_itemsets(baskets, onehot_df, minsup):
    engine = choose_engine(onehot_df)
    if engine == "fp_growth":
        itemsets = fpgrowth(onehot_df, min_support=minsup, use_colnames=True)
        itemsets.rename(columns={"itemsets":"itemsets"}, inplace=True)
    else:
        itemsets = eclat_frequent_itemsets(baskets, minsup)
        itemsets["itemsets"] = itemsets["itemsets"].apply(lambda s: frozenset(list(s)))
    return engine, itemsets

def rules_from_itemsets(itemsets, minconf):
    r = association_rules(itemsets, metric="confidence", min_threshold=minconf)
    return r.reset_index(drop=True)

# ---------- Scoring: strength + stability + profit ----------
def load_margins(dataset_dir):
    m = pd.read_csv(os.path.join(dataset_dir, "margins.csv"))
    return dict(zip(m["item"], m["margin_php"]))

def normalize(s):
    if len(s)==0: return s
    mn, mx = s.min(), s.max()
    if mx == mn: return s*0 + 1.0
    return (s - mn) / (mx - mn)

def rule_key(row):
    return (tuple(sorted(list(row["antecedents"]))), tuple(sorted(list(row["consequents"]))))

def score_rules(rules, margins, prev_keys=None):
    df = rules.copy()
    df["lift_c"] = df["lift"].clip(upper=3.0)
    df["support_n"] = normalize(df["support"])
    df["conf_n"] = normalize(df["confidence"])
    df["lift_n"] = normalize(df["lift_c"])

    df["key"] = df.apply(rule_key, axis=1)
    df["stability"] = df["key"].apply(lambda k: 1.0 if (prev_keys and k in prev_keys) else 0.0)

    df["profit"] = df["consequents"].apply(lambda c: sum(margins.get(i,0) for i in list(c)))
    df["profit_n"] = normalize(df["profit"])

    df["score"] = (
        0.28*df["support_n"] +
        0.30*df["conf_n"] +
        0.22*df["lift_n"] +
        0.10*df["stability"] +
        0.10*df["profit_n"]
    )
    return df.sort_values("score", ascending=False).reset_index(drop=True)

# ---------- Holdout Evaluation Loop (hit-rate) ----------
def holdout_hit_rate(rules_scored, test_baskets, top_n=40, min_occ=8):
    test_sets = [set(b) for b in test_baskets]
    out = []
    for _, r in rules_scored.head(top_n).iterrows():
        A = set(list(r["antecedents"]))
        C = set(list(r["consequents"]))
        occ = 0
        hit = 0
        for b in test_sets:
            if A.issubset(b):
                occ += 1
                if C.issubset(b):
                    hit += 1
        hr = (hit/occ) if occ > 0 else 0.0
        out.append({"key": r["key"], "occurrences": occ, "hits": hit, "hit_rate": hr})
    df = pd.DataFrame(out)
    # compute a reliable mean hit-rate (ignore tiny occurrences)
    reliable = df[df["occurrences"] >= min_occ]
    mean_hr = float(reliable["hit_rate"].mean()) if len(reliable) else 0.0
    return mean_hr, df

# ---------- Self-learning: auto threshold tuning (uses holdout score) ----------
def split_train_test(baskets, test_ratio=0.20):
    idx = np.arange(len(baskets))
    np.random.shuffle(idx)
    cut = int(len(baskets) * (1.0 - test_ratio))
    train_idx, test_idx = idx[:cut], idx[cut:]
    train = [baskets[i] for i in train_idx]
    test  = [baskets[i] for i in test_idx]
    return train, test

def tune(baskets, margins, prev_rules_scored=None):
    prev_keys = None
    if prev_rules_scored is not None and len(prev_rules_scored)>0:
        prev_keys = set(prev_rules_scored["key"].tolist())

    train_baskets, test_baskets = split_train_test(baskets, test_ratio=0.20)
    train_oh = onehot(train_baskets)

    best = None
    best_pack = None

    for minsup in [0.01, 0.015, 0.02, 0.03, 0.04]:
        engine, itemsets = mine_itemsets(train_baskets, train_oh, minsup)
        if len(itemsets)==0:
            continue

        for minconf in [0.20, 0.25, 0.30, 0.35, 0.45, 0.55]:
            rules = rules_from_itemsets(itemsets, minconf)
            if len(rules)==0:
                continue

            scored = score_rules(rules, margins, prev_keys)

            n_rules = len(scored)
            med_lift = float(scored["lift"].median())
            med_conf = float(scored["confidence"].median())

            # holdout evaluation loop (ML-like)
            mean_hr, _ = holdout_hit_rate(scored, test_baskets, top_n=35, min_occ=8)

            # penalties for too few/many rules
            penalty = 0.0
            if n_rules < 40: penalty += (40-n_rules)*0.015
            if n_rules > 140: penalty += (n_rules-140)*0.015

            # objective mixes rule quality + holdout hit-rate
            objective = (0.55*med_lift + 0.75*med_conf + 0.80*mean_hr + float(scored.head(15)["score"].mean())*1.0) - penalty

            if best is None or objective > best:
                best = objective
                best_pack = (engine, minsup, minconf, scored, mean_hr)

    engine, minsup, minconf, scored, mean_hr = best_pack
    return {"engine": engine, "minsup": minsup, "minconf": minconf, "rules_scored": scored, "objective": float(best), "holdout_mean_hr": float(mean_hr)}

# ---------- Drift detection (JS divergence on item distribution) ----------
def item_dist(onehot_df):
    freq = onehot_df.sum(axis=0).astype(float)
    p = (freq / max(freq.sum(), 1)).values
    return p, list(onehot_df.columns)

def drift_score(prev_onehot, curr_onehot):
    if prev_onehot is None:
        return {"drift": False, "js": 0.0}

    p_prev, cols_prev = item_dist(prev_onehot)
    p_curr, cols_curr = item_dist(curr_onehot)
    all_cols = sorted(set(cols_prev).union(cols_curr))

    def aligned(p, cols):
        d = dict(zip(cols, p))
        return np.array([d.get(c, 0.0) for c in all_cols], dtype=float)

    a = aligned(p_prev, cols_prev)
    b = aligned(p_curr, cols_curr)
    js = float(jensenshannon(a, b, base=2.0))
    return {"drift": js >= 0.12, "js": js}

# ---------- Recommendation services ----------
def fbt(scored, item, k=5):
    df = scored[scored["antecedents"].apply(lambda a: item in list(a))].head(200)
    df = df.sort_values("blend_score", ascending=False).head(k)
    out = []
    for _, r in df.iterrows():
        out.append({
            "with_item": item,
            "recommend": list(r["consequents"]),
            "blend_score": float(r["blend_score"]),
            "lift": float(r["lift"]),
            "confidence": float(r["confidence"])
        })
    return out

def cross_sell(scored, cart, k=3):
    cart = set(cart)
    df = scored[scored["antecedents"].apply(lambda a: set(list(a)).issubset(cart))].head(300)
    df = df.sort_values("blend_score", ascending=False)
    out = []
    for _, r in df.iterrows():
        cons = [x for x in list(r["consequents"]) if x not in cart]
        if cons:
            out.append({"cart": sorted(list(cart)), "add": cons, "blend_score": float(r["blend_score"])})
        if len(out) >= k:
            break
    return out

def promos(scored, k=6):
    df = scored[(scored["lift"]>=1.20) & (scored["confidence"]>=0.30)].head(400)
    df = df.sort_values(["profit","blend_score"], ascending=False).head(k)
    out = []
    for _, r in df.iterrows():
        bundle = list(r["antecedents"]) + list(r["consequents"])
        out.append({
            "bundle": bundle,
            "promo": "Bundle Discount 10%",
            "why": f"lift={r['lift']:.2f}, conf={r['confidence']:.2f}, profit≈{int(r['profit'])}php"
        })
    return out

def menu_rank(onehot_df, scored):
    pop = onehot_df.sum(axis=0).sort_values(ascending=False)
    impact = {}
    top = scored.head(250)
    for _, r in top.iterrows():
        for x in list(r["antecedents"]) + list(r["consequents"]):
            impact[x] = impact.get(x, 0.0) + float(r["blend_score"])

    df = pd.DataFrame({"item": pop.index, "pop": pop.values})
    df["impact"] = df["item"].map(impact).fillna(0.0)

    df["rank_score"] = (df["pop"]/max(df["pop"].max(),1)) + (df["impact"]/max(df["impact"].max(),1e-9))
    return df.sort_values("rank_score", ascending=False).reset_index(drop=True)

# ---------- Portfolio: Stable / Emerging / Fading ----------
def portfolio(prev_scored, curr_scored, top_k=25):
    if prev_scored is None or len(prev_scored)==0:
        return {"stable": [], "emerging": [k for k in curr_scored.head(top_k)["key"].tolist()], "fading": []}

    prev_top = set(prev_scored.head(top_k)["key"].tolist())
    curr_top = set(curr_scored.head(top_k)["key"].tolist())

    stable = list(prev_top.intersection(curr_top))
    emerging = list(curr_top - prev_top)
    fading = list(prev_top - curr_top)

    def k_to_dict(k):
        return {"antecedents": list(k[0]), "consequents": list(k[1])}

    return {
        "stable": [k_to_dict(k) for k in stable],
        "emerging": [k_to_dict(k) for k in emerging],
        "fading": [k_to_dict(k) for k in fading],
        "stable_count": len(stable),
        "emerging_count": len(emerging),
        "fading_count": len(fading)
    }

# ---------- Iteration runner (3 iterations + drift-adaptive blending) ----------
def run_dataset(dataset_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    margins = load_margins(dataset_dir)

    prev_long_rules = None
    prev_onehot = None
    prev_blended = None

    for it in [1,2,3]:
        df = load_batches(dataset_dir, it)
        baskets = parse_baskets(df)
        oh = onehot(baskets)

        # recent slice (last 300 tx)
        recent_df = df.tail(300).copy()
        recent_baskets = parse_baskets(recent_df)
        recent_oh = onehot(recent_baskets)

        drift = drift_score(prev_onehot, oh)

        # train long-term and recent models
        long_pack = tune(baskets, margins, prev_long_rules)
        recent_pack = tune(recent_baskets, margins, None)

        long_rules = long_pack["rules_scored"]
        recent_rules = recent_pack["rules_scored"]

        # drift-adaptive blending weights
        w_recent = 0.60 if drift["drift"] else 0.30
        w_long = 1.0 - w_recent

        recent_map = dict(zip(recent_rules["key"], recent_rules["score"]))
        blended = long_rules.copy()
        blended["recent_score"] = blended["key"].map(recent_map).fillna(0.0)
        blended["blend_score"] = (w_long*blended["score"]) + (w_recent*blended["recent_score"])

        # sort by blended score
        blended = blended.sort_values("blend_score", ascending=False).reset_index(drop=True)

        # portfolio vs previous iteration
        port = portfolio(prev_blended, blended, top_k=25)

        # export rules table
        export = blended.copy()
        export["antecedents"] = export["antecedents"].apply(lambda x: ", ".join(list(x)))
        export["consequents"] = export["consequents"].apply(lambda x: ", ".join(list(x)))
        export.to_csv(os.path.join(out_dir, f"iteration_{it}_rules.csv"), index=False)

        # export menu ranking
        menu = menu_rank(oh, blended)
        menu.to_csv(os.path.join(out_dir, f"iteration_{it}_menu_rank.csv"), index=False)

        # export recs JSON
        recs = {
            "iteration": it,
            "n_transactions": int(len(df)),
            "n_unique_items": int(oh.shape[1]),
            "drift": drift,
            "blend": {"w_long": w_long, "w_recent": w_recent},
            "engine_longterm": long_pack["engine"],
            "minsup_longterm": long_pack["minsup"],
            "minconf_longterm": long_pack["minconf"],
            "holdout_mean_hr_longterm": long_pack["holdout_mean_hr"],
            "engine_recent": recent_pack["engine"],
            "holdout_mean_hr_recent": recent_pack["holdout_mean_hr"],
            "fbt_burger": fbt(blended, "Burger", k=5),
            "cross_sell_burger": cross_sell(blended, ["Burger"], k=3),
            "promos": promos(blended, k=6),
            "portfolio": port
        }
        with open(os.path.join(out_dir, f"iteration_{it}_recs.json"), "w", encoding="utf-8") as f:
            json.dump(recs, f, indent=2)

        # bonus: segment-specific snapshot on Iteration 3 (adds uniqueness)
        if it == 3:
            seg_out = {}
            for seg in ["morning","lunch","dinner"]:
                seg_df = df[df["segment"] == seg]
                if len(seg_df) < 200:
                    continue
                seg_baskets = parse_baskets(seg_df)
                seg_pack = tune(seg_baskets, margins, None)
                seg_rules = seg_pack["rules_scored"].copy()
                seg_rules["blend_score"] = seg_rules["score"]  # single model inside segment
                seg_oh = onehot(seg_baskets)

                seg_out[seg] = {
                    "n_tx": int(len(seg_df)),
                    "engine": seg_pack["engine"],
                    "minsup": seg_pack["minsup"],
                    "minconf": seg_pack["minconf"],
                    "holdout_mean_hr": seg_pack["holdout_mean_hr"],
                    "menu_rank_top10": menu_rank(seg_oh, seg_rules).head(10).to_dict(orient="records"),
                    "promos_top3": promos(seg_rules, k=3)
                }

            with open(os.path.join(out_dir, f"iteration_{it}_segment_snapshot.json"), "w", encoding="utf-8") as f:
                json.dump(seg_out, f, indent=2)

        print(f"\n✅ {os.path.basename(dataset_dir)} ITER {it}")
        print("   drift:", drift, "| recent weight:", w_recent)
        print("   engines:", long_pack["engine"], "(long),", recent_pack["engine"], "(recent)")
        print("   holdout HR long:", long_pack["holdout_mean_hr"], "| recent:", recent_pack["holdout_mean_hr"])
        print("   portfolio:", {k: port.get(k) for k in ["stable_count","emerging_count","fading_count"]})

        # update prev
        prev_long_rules = long_rules
        prev_onehot = oh
        prev_blended = blended

def main():
    run_dataset("data/datasetA", "outputs/datasetA")
    run_dataset("data/datasetB", "outputs/datasetB")

if __name__ == "__main__":
    main()



✅ datasetA ITER 1
   drift: {'drift': False, 'js': 0.0} | recent weight: 0.3
   engines: fp_growth (long), fp_growth (recent)
   holdout HR long: 0.0 | recent: 0.0
   portfolio: {'stable_count': None, 'emerging_count': None, 'fading_count': None}

✅ datasetA ITER 2
   drift: {'drift': False, 'js': 0.027942930548365517} | recent weight: 0.3
   engines: fp_growth (long), fp_growth (recent)
   holdout HR long: 0.0 | recent: 0.0
   portfolio: {'stable_count': 0, 'emerging_count': 1, 'fading_count': 3}

✅ datasetA ITER 3
   drift: {'drift': False, 'js': 0.014569592549986292} | recent weight: 0.3
   engines: fp_growth (long), fp_growth (recent)
   holdout HR long: 0.0 | recent: 0.25
   portfolio: {'stable_count': 0, 'emerging_count': 1, 'fading_count': 1}

✅ datasetB ITER 1
   drift: {'drift': False, 'js': 0.0} | recent weight: 0.3
   engines: fp_growth (long), fp_growth (recent)
   holdout HR long: 0.27138391588519467 | recent: 0.0
   portfolio: {'stable_count': None, 'emerging_count': Non